# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(
    r"E:\Rafanelli\Belajar\AI\Reworkacademy\semantic-search-assignment\.env",
    override=True
)

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv(r'E:\Rafanelli\Belajar\AI\Reworkacademy\semantic-search-assignment\data\ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',  # sesuaikan versi model embedding
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [12]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

# Simpan index ke folder data.
index_path = r"E:\Rafanelli\Belajar\AI\Reworkacademy\semantic-search-assignment\data\ecommerce_faq.index"

faiss.write_index(index, index_path)

print("Index berhasil disimpan:", index_path)

# Baca kembali untuk memastikan file bisa digunakan.
saved_index = faiss.read_index(index_path)

print("Jumlah vektor:", saved_index.ntotal)
print("Dimensi vektor:", saved_index.d)



Jumlah vektor tersimpan di index: 59
Index berhasil disimpan: E:\Rafanelli\Belajar\AI\Reworkacademy\semantic-search-assignment\data\ecommerce_faq.index
Jumlah vektor: 59
Dimensi vektor: 3072


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353508)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.4088098)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [6]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [8]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.5-flash',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Halo! Tentu bisa. Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. 

Ada hal lain yang ingin kamu tanyakan?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [9]:
test_queries = [
    # 2 pertanyaan langsung dari FAQ
    "Metode pembayaran apa saja yang tersedia?",
    "Bagaimana cara melacak paket yang sedang dikirim?",

    # 2 pertanyaan dengan bahasa berbeda atau parafrase
    "Duit gue udah kepotong, kok pesanan masih belum terbayar?",
    "Barang yang datang beda dari pesanan, harus ngapain?",

    # 1 pertanyaan yang detail jawabannya tidak ada di FAQ
    "Berapa bunga cicilan kartu kredit BCA untuk tenor 12 bulan?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Metode pembayaran apa saja yang tersedia?
Jawaban: Halo! Kami mendukung beberapa metode pembayaran, yaitu:
- Transfer bank
- Kartu kredit/debit
- E-wallet
- Pembayaran melalui minimarket rekanan

Semoga membantu! Silakan hubungi kami kembali jika ada pertanyaan lain.
Sumber teratas: Metode pembayaran apa saja yang tersedia?
---
Query: Bagaimana cara melacak paket yang sedang dikirim?
Jawaban: Halo! 

Untuk melacak paket yang sedang dikirim, nomor resi akan muncul di halaman 'Pesanan Saya' setelah barang dikirim. Kamu dapat menggunakan nomor resi tersebut untuk melacak posisi paket di website kurir terkait.
Sumber teratas: Bagaimana cara melacak paket yang sedang dikirim?
---
Query: Duit gue udah kepotong, kok pesanan masih belum terbayar?
Jawaban: Halo! Kendala tersebut biasanya terjadi karena adanya keterlambatan sinkronisasi pada sistem pembayaran. 

Silakan tunggu sekitar 15-30 menit terlebih dahulu ya. Jika setelah itu status pesanan masih belum berubah, Anda dapat menghubun

In [10]:
hasil = semantic_search(
    "Barang yang datang beda dari pesanan, harus ngapain?"
)

for s in hasil:
    print("FAQ:", s["question"])
    print("Jawaban:", s["answer"])
    print("Distance:", s["distance"])
    print("---")

FAQ: Saya menerima barang yang salah, apa yang harus saya lakukan?
Jawaban: Ajukan komplain 'Barang Tidak Sesuai' melalui menu Pesanan Saya, sertakan foto barang yang diterima untuk mempercepat proses verifikasi.
Distance: 0.25013494
---
FAQ: Bagaimana prosedur pengembalian barang yang tidak sesuai?
Jawaban: Ajukan pengembalian melalui menu 'Pesanan Saya' dengan memilih alasan dan menyertakan bukti foto, maksimal 3 hari sejak barang diterima.
Distance: 0.3808969
---
FAQ: Apa yang harus dilakukan jika paket diterima dalam kondisi rusak?
Jawaban: Segera ambil foto/video kondisi paket saat diterima dan laporkan melalui menu komplain dalam waktu 2x24 jam sejak barang diterima.
Distance: 0.46660513
---


### Catatan Analisis Relevansi

Pengujian dilakukan menggunakan lima query: dua pertanyaan langsung,
dua parafrase, dan satu pertanyaan yang detail jawabannya tidak tersedia
dalam knowledge base. Sistem mengambil tiga FAQ terdekat sebagai konteks
untuk menghasilkan jawaban.

| No. | Query | Analisis |
|---|---|---|
| 1 | Metode pembayaran apa saja yang tersedia? | FAQ teratas sesuai. Jawaban menyebut transfer bank, kartu kredit/debit, e-wallet, dan minimarket rekanan sesuai sumber. |
| 2 | Bagaimana cara melacak paket yang sedang dikirim? | FAQ teratas sesuai. Jawaban menjelaskan lokasi nomor resi pada halaman Pesanan Saya dan penggunaannya di website kurir. |
| 3 | Duit gue udah kepotong, kok pesanan masih belum terbayar? | Sistem memahami bahasa percakapan dan menemukan FAQ tentang status pembayaran. Jawaban menjelaskan keterlambatan sinkronisasi serta waktu tunggu 15–30 menit sesuai sumber. |
| 4 | Barang yang datang beda dari pesanan, harus ngapain? | Sistem menemukan FAQ penerimaan barang yang salah. Jawaban menggabungkan informasi komplain, bukti foto, dan prosedur pengembalian dari sumber yang relevan. |
| 5 | Berapa bunga cicilan kartu kredit BCA untuk tenor 12 bulan? | FAQ teratas membahas cicilan secara umum, tetapi tidak memuat detail bunga BCA. LLM mengakui informasi belum tersedia dan menyarankan menghubungi customer service tanpa mengarang angka. |

Pada query keempat, pemeriksaan seluruh hasil retrieval menunjukkan:

| Peringkat | FAQ | Distance |
|---|---|---:|
| 1 | Saya menerima barang yang salah, apa yang harus saya lakukan? | 0,2501 |
| 2 | Bagaimana prosedur pengembalian barang yang tidak sesuai? | 0,3809 |
| 3 | Apa yang harus dilakukan jika paket diterima dalam kondisi rusak? | 0,4666 |

Detail batas waktu 3 hari dalam jawaban didukung FAQ kedua. Agar lebih
presisi, batas tersebut sebaiknya disebut khusus sebagai batas pengajuan
pengembalian barang. Ketentuan 2×24 jam pada FAQ paket rusak tidak
dicampurkan ke jawaban.

Keempat query dengan jawaban tersedia menemukan FAQ teratas yang sesuai,
termasuk dua parafrase. Pada satu query tanpa detail jawaban, LLM berhasil
menyatakan keterbatasan informasi. Hasil ini menunjukkan keberhasilan
pada sampel pengujian, tetapi belum cukup untuk menyimpulkan akurasi
sistem secara umum.

Keterbatasan yang ditemukan:
- Pengujian hanya mencakup lima query.
- Sistem belum menggunakan ambang distance untuk menyaring hasil yang
  kurang relevan.
- Tiga hasil retrieval dapat mencakup FAQ yang hanya berhubungan secara
  topik, seperti FAQ paket rusak pada pertanyaan barang salah.
- Distance bukan persentase keyakinan bahwa FAQ dapat menjawab query.
- Evaluasi jawaban harus mempertimbangkan seluruh konteks retrieval,
  bukan hanya FAQ teratas.
- Instruksi prompt belum menjamin semua jawaban berikutnya selalu bebas
  dari informasi yang tidak didukung sumber.